In [ ]:
#import variables 

import pandas as pd 
import os 

In [ ]:
#processing fNIRS timestamps

#extract start and end of trial and recording files for each dyad 

from datetime import datetime
import pandas as pd 
output_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Combined'
folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Start'
file_list = [file for file in os.listdir(folder_path) if file.endswith('.xlsx')]
for file in file_list:
    file_path = os.path.join(folder_path, file)
    if file.endswith('Recording.xlsx'):
        file_type = 'Recording.Start'
    elif file.endswith('Trial.xlsx'):
        file_type = 'Trial.Start'
    else:
        file_type = 'Unknown'
    df = pd.read_excel(file_path,header=None)
    df['Type'] = file_type
    if file.endswith('Recording.xlsx'):
        df = df[df['Type']=='Recording.Start']
        df = df.sort_values(by = df.columns[0],ascending=True)
        df = df.drop_duplicates(subset = df.columns[0],keep='last')
        df = df.sort_values(by = df.columns[0],ascending=True)
        if len(df) > 1:
           df = df.iloc[[1], :] 
    if file.endswith('Trial.xlsx'):
        df = df[df['Type']=='Trial.Start']
        df = df.drop_duplicates(subset = df.columns[0],keep='first')
    new_file_path = os.path.join(output_folder, os.path.splitext(file)[0] + '_modified.xlsx')
    df.to_excel(new_file_path, header=None, index=False)

folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/End'
file_list = [file for file in os.listdir(folder_path) if file.endswith('.xlsx')]
for file in file_list:
    file_path = os.path.join(folder_path, file)
    if file.endswith('Recording.xlsx'):
        file_type = 'Recording.End'
    elif file.endswith('Trial.xlsx'):
        file_type = 'Trial.End'
    else:
        file_type = 'Unknown'
    df = pd.read_excel(file_path,header=None)
    df['Type'] = file_type
    if file.endswith('Recording.xlsx'):
        df = df[df['Type']=='Recording.End']
        df = df.sort_values(by = df.columns[0],ascending=True)
        df = df.drop_duplicates(subset = df.columns[0],keep='first')
        df = df.sort_values(by = df.columns[0],ascending=True)
        if len(df) > 1:
            df = df.iloc[[1], :] 
    if file.endswith('Trial.xlsx'):
        df = df[df['Type']=='Trial.End']
        df = df.drop_duplicates(subset = df.columns[0],keep='first')
    new_file_path = os.path.join(output_folder, os.path.splitext(file)[0] + '_modified.xlsx')
    df.to_excel(new_file_path, header=None, index=False)

#concatenating start and end trials for each dyad 

FINAL_output_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/FINAL'
combined_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Combined'
dyad_max = 16
for dyad in range(1, dyad_max+1):
    dffirststart = pd.DataFrame()
    dfEnd = pd.DataFrame()
    for file in os.listdir(combined_folder):
     if file.endswith('_modified.xlsx'):
        file_parts = file.split('-')
        if len(file_parts) >= 3:
            file_dyad = int(file_parts[1])
            file_type = file_parts[2]
            if file_dyad == dyad:
                file_path = os.path.join(combined_folder, file)
                df = pd.read_excel(file_path, header=None)
                df.columns = ['Timestamp'] + [f'Column_{i}' for i in range(1, len(df.columns))]
                df = df[['Timestamp', 'Column_3']]
                if file_type == 'Start':
                    dffirststart = pd.concat([dffirststart, df], axis=0)
                elif file_type == 'End':
                    dfEnd = pd.concat([dfEnd, df], axis=0)
                dffirststart = dffirststart.drop_duplicates(subset=['Timestamp'], keep='first')
                dfEnd = dfEnd.drop_duplicates(subset=['Timestamp'], keep='first')
    FINAL = pd.concat([dffirststart, dfEnd], axis=0)
    FINAL.columns = ['FNIRS time', 'Event Type']
    FINAL.sort_values(by='FNIRS time', inplace=True)
    output_file = os.path.join(FINAL_output_folder, f'Dyad-{dyad}-FINAL.xlsx')
    FINAL.to_excel(output_file, index=False, header=True)

#extracting the correct times from the trigger file (UTC times) (saved to an 'actualtime' folder)

folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/FINAL'
file_list = [file for file in os.listdir(folder_path) if file.endswith('.txt')]

for file in file_list:
    file_parts = file.split('-')
    file_dyad = int(file_parts[1])
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path, delimiter=',', names=['Events', 'Server Time Date T0 UTC'], skiprows=1)
    df[['Event Type', 'DateTime']] = df['Events'].str.extract(r'([^0-9]+)(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})')
    df[['Date', 'Time']] = df['DateTime'].str.split(' ', expand=True)
    df['Time'] = df['Time'].str[:-4]
    df.drop(columns=['Events', 'DateTime'], inplace=True)
    df = df.iloc[:, 1:] 
    df.columns = ['Event Type','Date','Actual time']
    output_file_name = f'Dyad-{file_dyad}-actualtime.xlsx'
    df.to_excel(os.path.join(folder_path, output_file_name), index=False)

#extract fNIRS-based trigger click times (DYAD-X-FINAL) and real (UTC) trigger click times (-ACTUALTIME) for each trial and then combine into one dataframe (FINAL-NEW)

import os
import re
import pandas as pd

folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/FINAL'
file_pattern = re.compile(r"Dyad-\d+-FINAL\.xlsx") 
actual_time_pattern = re.compile(r"Dyad-(\d+)-actualtime\.xlsx")
final_new_pattern = re.compile(r"Dyad-(\d+)-FINAL-NEW\.xlsx")
actual_times = {}

for file in os.listdir(folder_path):
    final_new_match = file_pattern.match(file)
    if final_new_match:
        dyad_number = file.split('-')[1]  # Extract dyad number from the file name
        file_to_clip = os.path.join(folder_path, file)
        df = pd.read_excel(file_to_clip)
        recording_start_index = df[df['Event Type'] == 'Recording.Start'].index[0]
        df_clipped = df.iloc[recording_start_index:]
        df_clipped = df_clipped.dropna(how='all')
        df_clipped.to_excel(os.path.join(folder_path, f"Dyad-{dyad_number}-CHECK.xlsx"), index=False)
        for actual_time_file in os.listdir(folder_path):
            actual_time_match = actual_time_pattern.match(actual_time_file)
            if actual_time_match and actual_time_match.group(1) == dyad_number:
                actual_time_df = pd.read_excel(os.path.join(folder_path, actual_time_file))
                actual_time_df = actual_time_df.sort_values(by='Actual time')
                actual_time_df = actual_time_df.dropna(how='all')
                df_clipped.reset_index(drop=True, inplace=True)
                actual_time_df.reset_index(drop=True, inplace=True)
                merged_df = pd.concat([df_clipped, actual_time_df], axis=1)
                merged_df = merged_df.sort_values(by='Actual time')
                merged_df.to_csv(os.path.join(folder_path, f"Dyad-{dyad_number}-FINAL-MERGED.csv"), index=False)

In [ ]:
#create table to visualise fNIRS and Minecraft timestamps 

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import os
import re
import shutil

input_folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Trigger files/'
output_folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Trigger files relevant/'
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)
def calculate_new_number(participant_number):
    if participant_number < 1021:
        return participant_number - 1008
    else:
        return participant_number - 1010
for file_name in [f for f in os.listdir(input_folder_path) if f.endswith('.txt')]:
    match = re.match(r'.*?(\d+)_\d{2}-\d{2}-\d{2}_\d{2}-\w+-\d{4}', file_name)
    if match:
        participant_number = int(match.group(1))
        print(participant_number)
        new_number = calculate_new_number(participant_number)
        new_file_name = f'{new_number}.txt'
        input_file_path = os.path.join(input_folder_path, file_name)
        output_file_path = os.path.join(output_folder_path, new_file_name)
        shutil.copy2(input_file_path, output_file_path)
folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Trigger files relevant/'
file_list = [file for file in os.listdir(folder_path) if file.endswith('.txt')]
first_triggerdatetimeonly = []
end_triggerdatetimeonly = []
target_file = "1.txt"
for file in file_list:
  file_path = os.path.join(folder_path, file)
  if os.path.splitext(file)[0] == os.path.splitext(target_file)[0]:
    df = pd.read_csv(file_path, delimiter=',', names=['Events', 'Server Time Date T0 UTC'], skiprows=1)
    df[['Event Type', 'DateTime']] = df['Events'].str.extract(r'([^0-9]+)(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})')
    df[['Date', 'Time']] = df['DateTime'].str.split(' ', expand=True)
    df['Time'] = df['Time'].str[:-4]
    df.drop(columns=['Events', 'DateTime'], inplace=True)
    df = df.iloc[:, 1:] 
    output_file_path = os.path.join(folder_path, f'{os.path.splitext(file)[0]}.csv')
    df.to_csv(output_file_path, index=False)
    recording_start_time = pd.to_datetime(df.loc[df['Event Type'].str.contains('Recording.Start'), 'Time'].values[0])
    df['Time'] = pd.to_datetime(df['Time'].astype(str))
    df['Time Difference'] = (df['Time'] - recording_start_time).dt.total_seconds() 
    df['Event Type'] = df['Event Type'].str.strip()
    for event_type in df['Event Type'].unique():
      event_df = df[df['Event Type'] == event_type]
      if event_type == 'Recording.Start':
        color = 'blue'
      elif event_type == 'Trial.Start':
        color = 'green'
      elif event_type == 'Trial.End':
        color = 'red'
      elif event_type == 'Recording.End':
        color = 'purple'
      else:
        color = 'black'  
      plt.vlines(event_df['Time Difference'], ymin=0, ymax=1, colors=color, linestyles='solid', label=event_type)
      plt.xlim(0,max(event_df['Time Difference']))
    plt.yticks([])
    plt.xlabel('Time (seconds)')
    plt.title('Timeline of Events for File: {}'.format(file))
    plt.legend()
    plt.show()

In [ ]:
#double check there are no more 7 chase and follow trials for each dyad and ensure the trials are correctly labelled before then using this data to create the behavioural regressors

import pandas as pd 
FINALFRAME2 = pd.read_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEEVERYTHINGUSE.xlsx')
chase_events = FINALFRAME2[FINALFRAME2['Event'].str.contains('Chase')]
follow_events = FINALFRAME2[FINALFRAME2['Event'].str.contains('Follow')]
chase_event_counts = chase_events.groupby(['Dyad'])['Event'].nunique()
follow_event_counts = follow_events.groupby(['Dyad'])['Event'].nunique()
chasehere = pd.DataFrame(chase_event_counts)
followhere = pd.DataFrame(follow_event_counts)
chasehere.reset_index(inplace=True)
chasehere.rename(columns={'index': 'Dyad'}, inplace=True)
chasehere['Excess'] = chasehere['Event'].apply(lambda x: max(0, x - 7))
followhere.reset_index(inplace=True)
followhere.rename(columns={'index': 'Dyad'}, inplace=True)
followhere['Excess'] = followhere['Event'].apply(lambda x: max(0, x - 7))

#remove excess follow 

dyads_with_excessfollow = followhere[followhere['Excess'] > 0]
for index, row in dyads_with_excessfollow.iterrows():
    dyad = row['Dyad']
    excess = row['Excess']
    follow_events = FINALFRAME2[(FINALFRAME2['Dyad'] == dyad) & FINALFRAME2['Event'].str.contains('Follow')]
    excess_eventfollow = follow_events.groupby('Event').head(1)['Event'].iloc[:excess]
    excess_event_indices_follow = FINALFRAME2[(FINALFRAME2['Dyad'] == dyad) & (FINALFRAME2['Event'].str.contains('|'.join(excess_eventfollow)))].index
    FINALFRAME2.drop(excess_event_indices_follow, inplace=True)

#remove excess chase

dyads_with_excesschase = chasehere[chasehere['Excess'] > 0]
for index, row in dyads_with_excesschase.iterrows():
    dyad = row['Dyad']
    excess = row['Excess']
    chase_events = FINALFRAME2[(FINALFRAME2['Dyad'] == dyad) & FINALFRAME2['Event'].str.contains('Chase')]
    excess_eventchase = chase_events.groupby('Event').head(1)['Event'].iloc[:excess]
    excess_event_indices_chase = FINALFRAME2[(FINALFRAME2['Dyad'] == dyad) & (FINALFRAME2['Event'].str.contains('|'.join(excess_eventchase)))].index
    FINALFRAME2.drop(excess_event_indices_chase, inplace=True)

FINALFRAME2.to_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEEVERYTHINGUSECHECK.xlsx')

#Check new frame 

FINALFRAME2CHECK = pd.read_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEEVERYTHINGUSECHECK.xlsx')
chase_events_CHECK = FINALFRAME2CHECK[FINALFRAME2CHECK['Event'].str.contains('Chase')]
follow_events_CHECK = FINALFRAME2CHECK[FINALFRAME2CHECK['Event'].str.contains('Follow')]
chase_event_counts_CHECK = chase_events_CHECK.groupby(['Dyad'])['Event'].nunique()
follow_event_counts_CHECK = follow_events_CHECK.groupby(['Dyad'])['Event'].nunique()
chasehere_CHECK = pd.DataFrame(chase_event_counts_CHECK)
followhere_CHECK = pd.DataFrame(follow_event_counts_CHECK)
chasehere_CHECK.reset_index(inplace=True)
chasehere_CHECK.rename(columns={'index': 'Dyad'}, inplace=True)
chasehere_CHECK['Excess'] = chasehere_CHECK['Event'].apply(lambda x: max(0, x - 7))
followhere_CHECK.reset_index(inplace=True)
followhere_CHECK.rename(columns={'index': 'Dyad'}, inplace=True)
followhere_CHECK['Excess'] = followhere_CHECK['Event'].apply(lambda x: max(0, x - 7))

#Check new dataframe chase and follow events are labelled properly given removal above 

chase_events = FINALFRAME2CHECK[FINALFRAME2CHECK['Event'].str.startswith('Chase')]
follow_events = FINALFRAME2CHECK[FINALFRAME2CHECK['Event'].str.startswith('Follow')]
unique_chase_events = chase_events['Event'].unique()
unique_follow_events = follow_events['Event'].unique()
chase_event_mapping = {event: f'Chase {i+1}' for i, event in enumerate(unique_chase_events)}
follow_event_mapping = {event: f'Follow {i+1}' for i, event in enumerate(unique_follow_events)}
chase_events['Event'] = chase_events['Event'].map(chase_event_mapping)
follow_events['Event'] = follow_events['Event'].map(follow_event_mapping)
relabeled_events = pd.concat([chase_events, follow_events])
relabeled_events.to_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEEVERYTHINGUSECHECK.xlsx')

In [ ]:
#(RUN FROM HERE) create table with event onsets and durations from FINAL processed behavioural data and then merge this fNIRS data onsets 

#create table with event onsets and durations from FINAL processed behavioural data (ONSETTIMES)

import re
import pandas as pd 
import os
import datetime 
from datetime import datetime

output_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/FINALANALYSIS'
matrix = pd.read_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEEVERYTHINGUSECHECK.xlsx')
events_to_limit = ['Follow 72', 'Follow 77', 'Follow 78', 'Follow 86']
matrix = matrix[~((matrix['Event'].isin(events_to_limit)) & 
                                            (matrix['Length of trial'] > 60))]
matrix['Event_Type'] = matrix['Event'].str.extract(r'([a-zA-Z]+)')  
matrix['Event_Number'] = matrix['Event'].str.extract(r'(\d+)').astype(int)  
matrix = matrix.sort_values(by=['Event_Type', 'Event_Number']).reset_index(drop=True)
matrix['New_Event_Number'] = matrix.groupby('Event_Type')['Event_Number'].rank(method='dense').astype(int)
matrix['Event'] = matrix['Event_Type'] + ' ' + matrix['New_Event_Number'].astype(str)
matrix = matrix.drop(columns=['Event_Type', 'New_Event_Number', 'Event_Number'])
matrix.reset_index(drop=True, inplace=True)
matrix = matrix.sort_values(by=['Date', 'Time'])
matrixcol = matrix[['Date','Event','Length of trial','Time at start of chase/follow','Dyad']]
filtered_matrix = matrixcol.loc[matrix['Dyad'] >= 1]
filtered_matrix['Time at start of chase/follow'] = pd.to_datetime(filtered_matrix['Time at start of chase/follow'])
filtered_matrix = filtered_matrix.drop_duplicates(subset='Time at start of chase/follow', keep='first')
filtered_matrix['Time at start of chase/follow'] = filtered_matrix['Time at start of chase/follow'].dt.strftime('%H:%M:%S')
filtered_matrix = filtered_matrix.rename(columns={'Time at start of chase/follow': 'Event onset', 'Length of trial': 'Event duration'})
filtered_matrix.to_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Eventonsetsdurations.xlsx')

for dyad in filtered_matrix['Dyad'].unique():
    dyad_data = filtered_matrix[filtered_matrix['Dyad'] == dyad]
    dyad_data['Dyad'] = dyad_data['Dyad']
    dyad_data.columns = ['Date','Event','Event duration','Actual onset','Dyad']
    output_file = os.path.join(output_folder, f'Dyad-{dyad}-ONSETTIMES.xlsx')
    dyad_data.to_excel(output_file, index=False)

#merging the dataset with the trigger times (from trigger clicks)(merged) with that of the specific event types (from behaviour)(onsets), and label the events (e.g., chase 72). In the final frame, only the times which relate to a specific chase are kept and only the times from the trigger-clicks are kept (not the onset times from the Minecraft world). End file is named 'Merged_Dyad'. 

#rough code 

from datetime import datetime, timedelta

input_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/FINALANALYSIS'
folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/MATLABREADY'
MERGED_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/FINAL'
for file in os.listdir(input_folder):
    if 'ONSETTIMES' in file:
        onset_times_file = os.path.join(input_folder, file)
        dyad_number_onset = int(file.split('-')[1])  
        final_merged_file = os.path.join(MERGED_folder, f'Dyad-{dyad_number_onset}-FINAL-MERGED.csv')
        if os.path.isfile(final_merged_file):
            dyad_number_merged = int(final_merged_file.split('-')[1]) 
            if dyad_number_onset == dyad_number_merged:  
                onset_times_df = pd.read_excel(onset_times_file)
                if dyad_number_onset == 16:
                    onset_times_df['Actual onset'] = pd.to_datetime(onset_times_df['Actual onset'])
                    onset_times_df = onset_times_df[onset_times_df['Actual onset'] <= pd.to_datetime('10:31:26')]
                final_merged_df = pd.read_csv(final_merged_file)  
                final_merged_df.drop(columns=final_merged_df.columns[1], inplace=True)  
                final_merged_df.columns.values[1] = 'Event Type'  
                onset_times_df['Actual onset'] = pd.to_datetime(onset_times_df['Actual onset'])
                final_merged_df['Actual time'] = pd.to_datetime(final_merged_df['Actual time']) 
                combined_df = pd.DataFrame()     
                for index, row in onset_times_df.iterrows():
                    filtered_rows = final_merged_df[
                        (final_merged_df['Actual time'] >= row['Actual onset'] - timedelta(seconds=5)) &
                        (final_merged_df['Actual time'] <= row['Actual onset'] + timedelta(seconds=5))
                    ]
                    frameA = filtered_rows.reset_index(drop=True)
                    frameA = frameA[['FNIRS time','Date','Actual time','Event Type']]
                    frameB = pd.DataFrame(row).transpose().reset_index(drop=True)
                    frameB = frameB[['Event','Event duration','Actual onset','Dyad']]
                    frameA.dropna(inplace=True)
                    frameB.dropna(inplace=True)
                    combined_row = pd.concat([frameA, frameB], axis=1)
                    combined_row = combined_row.drop(columns=['Actual onset'])
                    combined_row.rename(columns={'Actual time': 'Onset time'}, inplace=True)
                    combined_df = pd.concat([combined_df, combined_row])
                    combined_df = combined_df.drop_duplicates(subset=['Onset time'])
                    combined_df = combined_df.sort_values(by='Onset time')
                    combined_df = combined_df.reset_index(drop=True)
                    time_diff = combined_df['Onset time'].diff()
                    one_second_apart = time_diff == pd.Timedelta(seconds=5)
                    mask = ~one_second_apart
                    combined_df = combined_df[mask]
                    columns_to_keep = ['FNIRS time', 'Date', 'Onset time','Event Type','Event', 'Event duration', 'Dyad']
                    combined_df = combined_df[columns_to_keep]
                    combined_df = combined_df[combined_df['Event Type'].str.contains('Trial.Start')]             
                combined_df.to_excel(os.path.join(folder_path, f'Merged_Dyad-{dyad_number_onset}-EYLATEST.xlsx'), index=False)     

#duplicating the dataset for each dyad, given that there should be one dataset per person. End file is named as 'triggers.xlsx'. 

import os
import shutil

folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/MATLABREADY'
destination_folder_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFiles'
for file in os.listdir(folder_path):
    if file.startswith('Merged_Dyad') and file.endswith('EYLATEST.xlsx'):
        dyad_number = int(file.split('_')[1].split('-')[1])
        participant_num = dyad_number*2-1
        participant_num2 = participant_num+1
        participant_num_str = str(participant_num).zfill(2)  
        participant_file = os.path.join(destination_folder_path, f'P{participant_num_str}_triggers.xlsx')
        participant_num_str2 = str(participant_num2).zfill(2)  
        participant_file2 = os.path.join(destination_folder_path, f'P{participant_num_str2}_triggers.xlsx')
        original_file_path = os.path.join(folder_path, file)
        df = pd.read_excel(original_file_path)
        df = df.dropna(subset=['Event'])
        df.to_excel(participant_file, index=False)
        df.to_excel(participant_file2, index=False)

#realign the fNIRS times given that we are clipping the data from 5 seconds before the first trial, meaning that the first event should say '5' in the fNIRS time column (5 seconds from the start of the fNIRS recording)

inputpath = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFiles'
outputpath = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFilesSHIFT'
file_list = [f for f in os.listdir(inputpath) if f.endswith('.xlsx')]
for file in file_list:
    filepath = os.path.join(inputpath, file)
    df = pd.read_excel(filepath)
    shift_value = df['FNIRS time'].iloc[0]-5
    df['FNIRS time'] = df['FNIRS time'] - shift_value
    df = df.dropna(subset=['Event'])
    filename, file_extension = os.path.splitext(file)
    output_file = os.path.join(outputpath, f"{filename}_shift{file_extension}")
    df.to_excel(output_file, index=False)

In [ ]:
#Remove unncessary columns from the participantfiles_shift dataframe

import os
import shutil
import pandas as pd 

input_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFilesSHIFT'
for file_name in os.listdir(input_path):
    if file_name.endswith('.xlsx'): 
        file_path = os.path.join(input_path, file_name)
        df = pd.read_excel(file_path)
        columns_to_remove = ['Lag_sensitivity_score', 'Pred_1_pred_2_average_distance_variation']
        df = df.drop(columns=[col for col in columns_to_remove if col in df.columns])
        df.to_excel(file_path, index=False) 

In [ ]:
#behavioural data with distances (not separated by chase or follow) for each person (output file is 'behaviour.xlsx')

import pandas as pd 
import numpy as np 
import os 

reshaped_df = pd.read_csv('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/FINALFRAMEFORMATRIX.csv')
reshaped_df.drop(columns=['Predator', 'Unnamed: 0','Time'], inplace=True)
reshaped_df.rename(columns={'Timeraw': 'Time', 'Dateraw': 'Date'}, inplace=True)
shiftdata_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFilesSHIFT/'
lagdata_path = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Lagdata/'
shiftdata_files = [f for f in os.listdir(shiftdata_path) if f.endswith('.xlsx')]
lag_files = [f for f in os.listdir(lagdata_path) if f.endswith('.xlsx')]
for file in shiftdata_files:
    participant_id = file.split('_')[0]  
    for lag in lag_files:
        participant_id2 = lag.split('_')[0]  
        if participant_id == participant_id2:
            shiftdata_df = pd.read_excel(os.path.join(shiftdata_path, file))
            shiftdata_df = shiftdata_df[[col for col in shiftdata_df.columns if 'Lag' not in col]]
            lagdata_df = pd.read_excel(os.path.join(lagdata_path, lag))
            lagdata_df = lagdata_df.groupby(['Dyad','Event']).agg({'Scaled self facing prey':'mean','Scaled self facing other':'mean','Distance between self and prey':'mean','Distance between self and other':'mean'})
            finaldata = pd.merge(shiftdata_df,lagdata_df,on=['Event','Dyad'])
            output_file = f"{participant_id}_triggers_shift.xlsx"
            finaldata.to_excel(os.path.join(shiftdata_path, output_file), index=False)
reshaped_df['Event'] = reshaped_df['Event'].str.replace('Chase', 'Hunt', case=False).str.replace('Follow', 'Follow', case=False)
participant_grouped = reshaped_df.groupby('Participant')
output_directory = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Preregressors/'
for participant, group_data in participant_grouped:
    participant_number = f'P{int(participant):02d}'
    new_file_name = f'{participant_number}-Preregressor.xlsx'
    output_file_path = os.path.join(output_directory, new_file_name)
    group_data.to_excel(output_file_path, index=False)

In [ ]:
#creating separate columns for chase and follow events and adding 0's in-between events in the behavioural data for the chase (when follow is present) and follow (when chase is present) events. Output is 'FINALFRAME-'. 

import os
import pandas as pd
import re 
import numpy as np 

#create columns for chase and follow distances 

correlation_matrices = []
inputpath = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Preregressors/'
outputpath_bothpredators = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Regressorsbothpredators/'
correlationpath = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/Behaviouraldata/Behaviouralstatsanalysis'
file_list = [f for f in os.listdir(inputpath) if f.endswith('Preregressor.xlsx')]
for file in file_list:
    filepath = os.path.join(inputpath, file)
    participant_id = file.split('-')[0]  
    number = re.search(r'\d+', participant_id).group()
    participant_idn = int(number)
    df = pd.read_excel(filepath)
    df['Participant'] = participant_idn
    if participant_idn % 2 == 0:
        dyad_number = participant_idn // 2
    else:
        dyad_number = (participant_idn + 1) // 2
    df['Dyad'] = dyad_number
    chase_data = df[df['Event'].str.contains('Hunt')]
    follow_data = df[df['Event'].str.contains('Follow')]
    chase_data.columns = [col + ' hunt' if col not in ['Date', 'Time', 'Event','Dyad','Participant'] else col for col in chase_data.columns]
    follow_data.columns = [col + ' follow' if col not in ['Date', 'Time', 'Event','Dyad','Participant'] else col for col in follow_data.columns]
    merged_data = pd.merge(chase_data, follow_data, how='outer', on=['Date', 'Time', 'Event','Participant','Dyad'])
    merged_data.fillna(0, inplace=True)
    filename = f'FINALFRAME_{participant_id}.xlsx'
    merged_data = merged_data.sort_values(by='Time')
    merged_data['Time'] = pd.to_datetime(merged_data['Time'])
    start_time = merged_data['Time'].min()
    end_time = merged_data['Time'].max()
    all_times = pd.date_range(start=start_time, end=end_time, freq='s')
    all_data = pd.DataFrame({'Time': all_times})
    merged_data = pd.merge(all_data, merged_data, how='left', on='Time').fillna(0)
    merged_data = merged_data[merged_data['Date']!='0']
    merged_data['Date'] = merged_data['Date'].astype(str)
    merged_data['Date'] = merged_data['Date'].str.split().str[0]
    merged_data['Date'] = pd.to_datetime(merged_data['Date'], errors='coerce').dt.date
    columns_to_replace = ['Dyad', 'Date', 'Participant']
    merged_data[columns_to_replace] = merged_data[columns_to_replace].replace(0, np.nan)
    merged_data['Event'] = merged_data['Event'].replace(0, 'in-between trials')
    merged_data.fillna(method='ffill', inplace=True)
    sequence = range(1, len(merged_data) + 1)
    merged_data['Timechase'] = sequence
    merged_data['Timefollow'] = sequence
    new_rows_columns = merged_data.columns.tolist()
    new_rows_data = {col: [0] * 5 for col in new_rows_columns}
    new_rows_data['Event'] = ['Pre-trial period'] * 5
    new_rows_data['Time'] = pd.date_range(start='1900-01-01 18:44:15', end='1900-01-01 18:44:19', freq='s').time
    new_rows = pd.DataFrame(new_rows_data)
    new_rows[['Date', 'Dyad', 'Participant']] = merged_data[['Date', 'Dyad', 'Participant']].iloc[0]
    merged_data = pd.concat([new_rows, merged_data], ignore_index=True)
    merged_data.reset_index(drop=True, inplace=True)
    merged_data['Time'] = merged_data['Time'].apply(lambda x: x.strftime('%H:%M:%S'))
    merged_data.fillna(0, inplace=True)
    output_file_path_bothpredators = os.path.join(outputpath_bothpredators, filename)
    merged_data.to_excel(output_file_path_bothpredators, index=False)

In [ ]:
#remove any files currently in the 'RegressorsbothpredatorsFINAL' folder

import os 
import glob

folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/RegressorsbothpredatorsFINAL/'
files = glob.glob(os.path.join(folder, '*'))
for file in files:
    os.remove(file)

In [ ]:
#check regressors.mat durations match onsets file durations 

import os
import pandas as pd

regressors_folder3 = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Regressorsbothpredators/'
output_folder3 = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/RegressorsbothpredatorsFINAL/'
participant_files_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFilesSHIFT/'

def process_files(regressors_folder, output_folder):
    for file1 in os.listdir(regressors_folder):
        if file1.endswith('.xlsx'):
            file_path1 = os.path.join(regressors_folder, file1)
            file_name1 = os.path.basename(file_path1)
            if 'FINALFRAME' in file_name1:
                PNdf1 = file_name1.split('_')[1].split('.')[0]
            else:
                PNdf1 = None
            df1 = pd.read_excel(file_path1)
            for file2 in os.listdir(participant_files_folder):
                if file2.startswith('P') & file2.endswith('.xlsx'):
                    file_path2 = os.path.join(participant_files_folder, file2)
                    df2 = pd.read_excel(file_path2)
                    file_name2 = os.path.basename(file_path2)
                    if 'triggers_shift' in file_name2:
                        PNdf2 = file_name2.split('_')[0]
                    if PNdf1 == PNdf2:
                        unique_events_df1 = df1['Event'].unique()
                        unique_events_df2 = df2['Event'].unique()
                        event_of_interest = []
                        duration = []
                        start_row = [] 
                        for event in unique_events_df1:
                            event_data = df1[df1['Event'] == event]
                            event_of_interest.append(event)
                            duration.append(len(event_data))
                            start_row.append(event_data.index[0])  
                        result_df1 = pd.DataFrame({
                            'Event_of_interest': event_of_interest,
                            'Duration': duration,
                            'Start_row': start_row
                        })
                        unique_events_df1_results = result_df1['Event_of_interest'].unique()
                        for j in unique_events_df1_results:
                            if j in unique_events_df2 and ('Hunt' in j or 'Follow' in j):
                                duration_df2 = df2[df2['Event'] == j]['Event duration'].iloc[0]
                                duration_df1 = result_df1[result_df1['Event_of_interest'] == j]['Duration'].iloc[0]
                                if duration_df2 != duration_df1:
                                    num_rows_to_remove = abs(int(duration_df2 - duration_df1))
                                    condition = df1['Event'] == j
                                    indices_to_drop = df1[condition].tail(num_rows_to_remove).index
                                    df1 = df1.drop(indices_to_drop).reset_index(drop=True)  
                        output_file_path = os.path.join(output_folder, file1)
                        wanted_columns = ['Dyad','Participant','Date','Time','Event','predfront hunt','predback hunt','prednear hunt','predfar hunt','predfront follow','predback follow','prednear follow','predfar follow','preyaheadpred hunt','preysidepred hunt','preybehindpred hunt','preybehindpred follow','preysidepred follow','preyaheadpred follow','predpreynear hunt','predpreymid hunt','predpreyfar hunt','predpreynear follow','predpreymid follow','predpreyfar follow','preyblockfirst hunt','preyblocksecond hunt','preyblockthird hunt','preyblockfourth hunt','preyblockfifth hunt','preyblocksixth hunt','preyfrontpred hunt','preyrightpred hunt','preyleftpred hunt','Scaled self facing prey hunt','Scaled self facing prey2 hunt','Scaled self facing other hunt','Scaled self facing other2 hunt','Scaled self facing other2 follow','Scaled self facing prey2 follow','Scaled self facing other follow','Lag Score hunt','Lag Score follow','Correlation in movement between the two predators hunt','Correlation in movement between the two predators follow','Distance between self and prey hunt','Second-to-second change in distance between self and prey hunt','Distance between self and other hunt','Distance between self and other follow','Second-to-second change in distance between self and other hunt','Second-to-second change in distance between self and other follow','Intersection angle between predators and prey hunt','Average second-to-second change in intersection angle between predators and prey hunt','Distance between self and prey follow','Second-to-second change in distance between self and prey follow','Intersection angle between predators and prey follow','Average second-to-second change in intersection angle between predators and prey follow','Scaled self facing prey follow','Path distance travelled by the self hunt','Path distance travelled by the self follow']
                        df1 = df1[wanted_columns]
                        df1.to_excel(output_file_path, index=False)

process_files(regressors_folder3, output_folder3)

In [ ]:
#Create individual dataframe for averaged breathing data/combined breathing data 

import shutil
import os
import pandas as pd 
import numpy as np 
from scipy.stats import zscore

physio_input = pd.read_csv('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/Biopac/FILTERED_ORIGINALDATABASELINESUBTRACTEDBOTHPRED.csv')
predator1_columns = [col for col in physio_input.columns if '_predator1' in col]
predator2_columns = [col for col in physio_input.columns if '_predator2' in col]
df_predator1 = physio_input[['Dyad', 'Time', 'Event'] + predator1_columns].copy()
df_predator1.columns = df_predator1.columns.str.replace('_predator1', '')
df_predator1['Predator'] = 1
df_predator2 = physio_input[['Dyad', 'Time', 'Event'] + predator2_columns].copy()
df_predator2.columns = df_predator2.columns.str.replace('_predator2', '')
df_predator2['Predator'] = 2
combined_df = pd.concat([df_predator1, df_predator2], ignore_index=True)
def assign_participant(row):
    return (row['Dyad'] - 1) * 2 + row['Predator']
combined_df['Participant'] = combined_df.apply(assign_participant, axis=1)
combined_df = combined_df.drop(columns=['Predator'])
combined_df['Time'] = pd.to_datetime(combined_df['Time'])
combined_df['Time'] = combined_df['Time'].dt.time
combined_df = combined_df.sort_values(by=['Participant', 'Time']).reset_index(drop=True)
input_folder1 = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/RegressorsbothpredatorsFINAL/'
destination_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/RegressorsbothpredatorsFINALFORFNIRS/'
os.makedirs(destination_folder, exist_ok=True)
column_mapping = {
    'Dyad': 'Dyad',
    'Participant': 'Participant',
    'Date': 'Date',
    'Time': 'Time',
    'Event': 'Event',
    'Lag Score hunt': 'LagScoreHunt',
    'Lag Score follow': 'LagScoreFollow',
    'Correlation in movement between the two predators hunt': 'MovementCorrHunt',
    'Correlation in movement between the two predators follow': 'MovementCorrFollow',
    'Distance between self and prey hunt': 'PreyDistHunt',
    'Second-to-second change in distance between self and prey hunt': 'DeltaPreyDistHunt',
    'Distance between self and other hunt': 'OtherDistHunt',
    'Distance between self and other follow': 'OtherDistFollow',
    'Second-to-second change in distance between self and other hunt': 'DeltaOtherDistHunt',
    'Second-to-second change in distance between self and other follow': 'DeltaOtherDistFollow',
    'Intersection angle between predators and prey hunt': 'IntersectAngleHunt',
    'Average second-to-second change in intersection angle between predators and prey hunt': 'DeltaIntersectAngleHunt',
    'Scaled self facing prey hunt': 'DirectionToPreyHunt',
    'Scaled self facing prey follow': 'DirectionToPreyFollow',
    'Scaled self facing other hunt': 'DirectionToOtherHunt',
    'Scaled self facing other follow': 'DirectionToOtherFollow',
    'Scaled self facing prey2 hunt': 'DirectionToPreyHunt2',
    'Scaled self facing prey2 follow': 'DirectionToPreyFollow2',
    'Scaled self facing other2 hunt': 'DirectionToOtherHunt2',
    'Scaled self facing other2 follow': 'DirectionToOtherFollow2',
    'Distance between self and prey follow': 'PreyDistFollow',
    'Second-to-second change in distance between self and prey follow': 'DeltaPreyDistFollow',
    'Intersection angle between predators and prey follow': 'IntersectAngleFollow',
    'Average second-to-second change in intersection angle between predators and prey follow': 'DeltaIntersectAngleFollow',
    'Path distance travelled by the self hunt': 'PathDistHunt',
    'Path distance travelled by the self follow': 'PathDistFollow',
    'HRBPM': 'HRBPM',
    'BRRSP_Rate': 'BRRSPRate',
    'SCREDA_Tonic': 'ScredaTonic',
    'preyaheadpred hunt':'preyaheadhunt',
    'preysidepred hunt':'preysidehunt',
    'preybehindpred hunt':'preybehindhunt',
    'preyaheadpred follow':'preyaheadfollow',
    'preysidepred follow':'preysidefollow',
    'preybehindpred follow':'preybehindfollow',
    'predpreynear hunt':'predpreynearhunt',
    'predpreymid hunt':'predpreymidhunt',
    'predpreyfar hunt':'predpreyfarhunt',
    'predpreynear follow':'predpreynearfollow',
    'predpreymid follow':'predpreymidfollow',
    'predpreyfar follow':'predpreyfarfollow',
    'predothernear hunt':'predpreynearhunt',
    'predothermid hunt':'predpreymidhunt',
    'predotherfar hunt':'predpreyfarhunt',
    'predothernear follow':'predpreynearfollow',
    'predothermid follow':'predpreymidfollow',
    'predotherfar follow':'predpreyfarfollow',
    'preyfrontpred hunt':'preyfrontpredhunt',
    'preyleftpred hunt':'preyleftpredhunt',
    'preyrightpred hunt':'preyrightpredhunt',
    'preyblockfirst hunt':'preyblockfirsthunt',
    'preyblocksecond hunt':'preyblocksecondhunt',
    'preyblockthird hunt':'preyblockthirdhunt',
    'preyblockfourth hunt':'preyblockfourthhunt',
    'preyblockfifth hunt':'preyblockfifthhunt',
    'preyblocksixth hunt':'preyblocksixthhunt',
    'predfront hunt':'predfronthunt',
    'predback hunt':'predbackhunt',
    'predfront follow':'predfrontfollow',
    'predback follow':'predbackfollow',
    'prednear hunt':'prednearhunt',
    'predfar hunt':'predfarhunt',
    'prednear follow':'prednearfollow',
    'predfar follow':'predfarfollow'
}
wanted_columns = [
    'Dyad',
    'Participant',
    'Date',
    'Time',
    'Event',
    'preyaheadhunt',
    'preysidehunt',
    'preybehindhunt',
    'preyaheadfollow',
    'preysidefollow',
    'preybehindfollow',
    'predpreynearhunt',
    'predpreymidhunt',
    'predpreyfarhunt',
    'predpreynearfollow',
    'predpreymidfollow',
    'predpreyfarfollow',
    'DirectionToPreyHunt',
    'DirectionToPreyFollow',
    'DirectionToOtherHunt',
    'DirectionToOtherFollow',
    'PreyDistHunt',
    'PreyDistFollow',
    'OtherDistHunt',
    'OtherDistFollow',
    'prednearhunt',
    'prednearfollow',
    'predfarhunt',
    'predfarfollow',
    'predfronthunt',
    'predfrontfollow',
    'predbackhunt',
    'predbackfollow',
    'HRBPM',
    'BRRSPRate'
]
all_files = os.listdir(input_folder1)
odd_files = [file for file in all_files if file.startswith('FINALFRAME_P') and file.endswith('.xlsx')]
for file in all_files:
    if file not in odd_files:
        file_path = os.path.join(input_folder1, file)
        os.remove(file_path)
for filename in odd_files:
    source_file = os.path.join(input_folder1, filename)
    df = pd.read_excel(source_file)
    dyad_number = df['Dyad'].iloc[0]
    participant_number = df['Participant'].iloc[0]
    physio_data_dyad = combined_df[(combined_df['Dyad'] == dyad_number) & (combined_df['Participant'] == participant_number)]
    df['Time'] = pd.to_datetime(df['Time']).dt.time
    merged_data = pd.merge(df, physio_data_dyad[['Dyad','Participant','Time','HRBPM','BRRSP_Rate','SCREDA_Tonic']], on=['Dyad','Participant', 'Time'], how='left')
    merged_data = merged_data.rename(columns=column_mapping)
    merged_data = merged_data[wanted_columns]
    merged_data.rename(columns={'Timechase': 'Timehunt'}, inplace=True)
    merged_data = merged_data.drop(columns=['Timehunt', 'Timefollow'], errors='ignore')
    destination_file = os.path.join(destination_folder, filename)
    merged_data.loc[:4, ['HRBPM', 'BRRSPRate']] = 0
    merged_data[['HRBPM', 'BRRSPRate']] = merged_data[['HRBPM', 'BRRSPRate']].fillna(method='ffill')
    merged_data = merged_data.rename(columns={'Time': 'Timeoriginal'})
    columns_to_zscore = merged_data.columns.difference(['Dyad', 'Participant','Timeoriginal','Date', 'Event','prednearhunt','predfarhunt','predfronthunt','predbackhunt','prednearfollow','predfarfollow','predfrontfollow','predbackfollow','preyaheadhunt','preysidehunt','preybehindhunt','preyaheadfollow','preysidefollow','preybehindfollow','predpreynearhunt','predpreymidhunt','predpreyfarhunt','predpreynearfollow','predpreymidfollow','predpreyfarfollow'])
    merged_data = merged_data.drop(columns=['Time'], errors='ignore')
    merged_data['Time'] = range(1, len(merged_data) + 1)
    newfolder = '/Users/macbook/Desktop/testfolder/'
    destination_filetest = os.path.join(newfolder, filename)
    merged_data.to_excel(destination_filetest, index=False)
    merged_data[columns_to_zscore] = merged_data[columns_to_zscore].apply(zscore)
    merged_data['Time'] = zscore(merged_data['Time'])
    if participant_number in [31, 32]:
        hunt_98_indices = merged_data[merged_data['Event'] == 'Hunt 98'].index
        if len(hunt_98_indices) > 0:
           last_hunt_98_index = hunt_98_indices[-1] 
           merged_data = merged_data.iloc[:last_hunt_98_index + 1]
    merged_data['predpreynear'] = merged_data['predpreynearhunt'] + merged_data['predpreynearfollow']
    merged_data['predpreymid'] = merged_data['predpreymidhunt'] + merged_data['predpreymidfollow']
    merged_data['predpreyfar'] = merged_data['predpreyfarhunt'] + merged_data['predpreyfarfollow']
    merged_data['preyahead'] = merged_data['preyaheadhunt'] + merged_data['preyaheadfollow']
    merged_data['preyside'] = merged_data['preysidehunt'] + merged_data['preysidefollow']
    merged_data['preybehind'] = merged_data['preybehindhunt'] + merged_data['preybehindfollow']
    merged_data['prednear'] = merged_data['prednearhunt'] + merged_data['prednearfollow']
    merged_data['predfar'] = merged_data['predfarhunt'] + merged_data['predfarfollow']
    merged_data['predfront'] = merged_data['predfronthunt'] + merged_data['predfrontfollow']
    merged_data['predback'] = merged_data['predbackhunt'] + merged_data['predbackfollow']
    merged_data[['Dyad','Participant','Date','Time','Event','HRBPM','BRRSPRate']].to_excel(destination_file, index=False)
    merged_data['FNIRS time'] = merged_data.groupby('Date').cumcount()
    merged_datahunt = merged_data[merged_data['Event'].str.contains('Hunt')][['Dyad','FNIRS time','prednearhunt','predfarhunt','predfronthunt','predbackhunt','DirectionToPreyHunt','DirectionToOtherHunt','PreyDistHunt','OtherDistHunt','predpreynearhunt','predpreymidhunt','predpreyfarhunt']]
    merged_datahunt['Duration'] = 1 
    merged_datafollow = merged_data[merged_data['Event'].str.contains('Follow')][['Dyad','FNIRS time','prednearfollow','predfarfollow','predfrontfollow','predbackfollow','DirectionToPreyFollow','DirectionToOtherFollow','PreyDistFollow','OtherDistFollow','predpreynearfollow','predpreymidfollow','predpreyfarfollow']]
    merged_datafollow['Duration'] = 1 
    merged_datahunt.to_excel(f'/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Huntparametrics/P{participant_number:02}_Huntparametrics.xlsx')
    merged_datafollow.to_excel(f'/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Followparametrics/P{participant_number:02}_Followparametrics.xlsx')
    merged_datahunt = merged_datahunt.rename(columns={'prednearhunt':'prednear','predfarhunt':'predfar','predfronthunt':'predfront','predbackhunt':'predback','DirectionToPreyHunt':'DirectionToPrey','DirectionToOtherHunt':'DirectionToOther','PreyDistHunt':'PreyDist','OtherDistHunt':'OtherDist','predpreynearhunt':'predpreynear','predpreymidhunt':'predpreymid','predpreyfarhunt':'predpreyfar'})
    merged_datafollow = merged_datafollow.rename(columns={'prednearfollow':'prednear','predfarfollow':'predfar','predfrontfollow':'predfront','predbackfollow':'predback','DirectionToPreyFollow':'DirectionToPrey','DirectionToOtherFollow':'DirectionToOther','PreyDistFollow':'PreyDist','OtherDistFollow':'OtherDist','predpreynearfollow':'predpreynear','predpreymidfollow':'predpreymid','predpreyfarfollow':'predpreyfar'})
    merged_finalframe = pd.concat([merged_datahunt,merged_datafollow],axis=0)
    merged_finalframe.to_excel(f'/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Bothparametrics/P{participant_number:02}_combinedparametrics.xlsx')

In [ ]:
#Process just the behavioural data

import os
import pandas as pd
from datetime import datetime, timedelta
import shutil

debriefing_folder_A = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Debrief scoring participant A'
debriefing_folder_B = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Debrief scoring participant B'
destination_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/RegressorsbothpredatorsFINALFORFNIRS/'
renamed_debrief_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Renameddebrief'

def parse_datetime_from_filename(filename):
    parts = filename.split(' - ')
    if len(parts) > 3:
        date_str = parts[2]  
        time_str = parts[3][:6] 
        dt = datetime.strptime(date_str + time_str, '%d%m%Y%H%M%S')
        dt_plus_10 = dt + timedelta(minutes=10)
        return dt_plus_10.strftime('%Y-%m-%d'), dt_plus_10.strftime('%H:%M:%S')
    return None, None

renamed_files = set()
for debriefing_folder in [debriefing_folder_A, debriefing_folder_B]:
    for debrief_filename in os.listdir(debriefing_folder):
        if debrief_filename.endswith('.xlsx'):
            debrief_filepath = os.path.join(debriefing_folder, debrief_filename)
            debrief_date, debrief_time = parse_datetime_from_filename(debrief_filename)
            for dest_filename in os.listdir(destination_folder):
                if dest_filename.endswith('.xlsx'):
                    dest_filepath = os.path.join(destination_folder, dest_filename)
                    dest_df = pd.read_excel(dest_filepath)
                    dest_df['Date'] = pd.to_datetime(dest_df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
                    dest_df['Timeoriginal'] = pd.to_datetime(dest_df['Timeoriginal'], format='%H:%M:%S', errors='coerce').dt.strftime('%H:%M:%S')
                    dest_df['Date'] = dest_df['Date'].str.strip()
                    dest_df['Timeoriginal'] = dest_df['Timeoriginal'].str.strip()
                    dest_df.columns = dest_df.columns.str.strip()                   
                    participant_number = dest_df['Participant'].iloc[0]
                    debrief_date = debrief_date.strip()
                    debrief_time = debrief_time.strip()
                    matching_row = dest_df[(dest_df['Date'] == debrief_date) & (dest_df['Timeoriginal'] == debrief_time)]
                    if not matching_row.empty:
                      if 'pred2' in debrief_filename:
                            if participant_number % 2 == 0:
                                new_filename = f"Debriefing_{participant_number}.xlsx"
                            else:
                                new_filename = f"Debriefing_{participant_number + 1}.xlsx"
                      else:
                            if participant_number % 2 != 0:
                                new_filename = f"Debriefing_{participant_number}.xlsx"
                            else:
                                new_filename = f"Debriefing_{participant_number - 1}.xlsx"
                      new_filepath = os.path.join(renamed_debrief_folder, new_filename)
                      shutil.copy(debrief_filepath, new_filepath)

def count_files(folder_path):
    return len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])

files_in_A = count_files(debriefing_folder_A)
files_in_B = count_files(debriefing_folder_B)
files_in_renamed = count_files(renamed_debrief_folder)
print(f"Mismatch! Files in A: {files_in_A}, Files in B: {files_in_B}, Files in Renamed Folder: {files_in_renamed}")

renamed_debrief_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Renameddebrief'
target_columns = [
    'Planning', 'Surprise',
    'Expectation confirmation',
    'Theory of Mind', 'Happy',
    'Frustrated'
]
summary_data = []
for filename in os.listdir(renamed_debrief_folder):
    if filename.endswith('.xlsx'):  
        file_path = os.path.join(renamed_debrief_folder, filename)
        df = pd.read_excel(file_path)
        df.columns = df.columns.str.strip()
        if 'Spatial planning route' in df.columns:
            df.rename(columns={'Spatial planning route': 'Planning'}, inplace=True)
        if 'Simulation outcome - expectation confirmation' in df.columns:
            df.rename(columns={'Simulation outcome - expectation confirmation': 'Expectation confirmation'}, inplace=True)
        if 'Theory of Mind - Prey' in df.columns and 'Theory of Mind - Predator' in df.columns:
            df['Theory of Mind'] = df[['Theory of Mind - Prey', 'Theory of Mind - Predator']].apply(lambda x: 1 if x.any() == 1 else 0, axis=1)
            df.drop(columns=['Theory of Mind - Prey', 'Theory of Mind - Predator'], inplace=True)
        if 'Simulation outcome - surprise close' in df.columns and 'Simulation outcome - surprise far' in df.columns:
            df['Surprise'] = df[['Simulation outcome - surprise close', 'Simulation outcome - surprise far']].apply(lambda x: 1 if x.any() == 1 else 0, axis=1)
            df.drop(columns=['Simulation outcome - surprise close', 'Simulation outcome - surprise far'], inplace=True)
        column_sums = df[target_columns].sum(numeric_only=True)
        participant_id = filename.split('_')[-1].split('.')[0]  
        summary_data.append({'Participant': participant_id, **column_sums})
        modified_file_path = os.path.join(renamed_debrief_folder, filename)
        final_columns = [
    'Planning', 'Expectation confirmation', 'Happy', 
    'Frustrated', 'Date', 'Event', 'Time (secs)', 
    'Theory of Mind', 'Surprise'
        ]
        df = df[final_columns]
        df.to_excel(modified_file_path, index=False)
final_df = pd.DataFrame(summary_data)
final_df = final_df.fillna(0)
output_path = os.path.join(renamed_debrief_folder, 'summary_scores.xlsx')
final_df = final_df.loc[:, ~(final_df.eq(0).all())]
final_df = final_df[~final_df['Participant'].str.contains('scores', case=False, na=False)]
final_df.to_excel('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Debriefsummaryscores/summary_scores.xlsx', index=False)

renamed_debrief_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Renameddebrief'
renamed_debriefpairs_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Renameddebriefpairs'
files = [f for f in os.listdir(renamed_debrief_folder) if f.endswith('.xlsx')]
for file in files:
    source_path = os.path.join(renamed_debrief_folder, file)
    destination_path = os.path.join(renamed_debriefpairs_folder, file)
    shutil.move(source_path, destination_path)
files_in_pairs_folder = os.listdir(renamed_debriefpairs_folder)
expected_pairs = [(i, i+1) for i in range(1, 32, 2)] 
files_to_remove = []
for odd, even in expected_pairs:
    file_odd = f"Debriefing_{odd}.xlsx"
    file_even = f"Debriefing_{even}.xlsx"
    if file_odd in files_in_pairs_folder and file_even not in files_in_pairs_folder:
        files_to_remove.append(file_odd)  # Remove odd file if even doesn't exist
    elif file_even in files_in_pairs_folder and file_odd not in files_in_pairs_folder:
        files_to_remove.append(file_even)  # Remove even file if odd doesn't exist
for file in files_to_remove:
    unmatched_filepath = os.path.join(renamed_debriefpairs_folder, file)
    os.remove(unmatched_filepath)
    print(f"Removed unmatched file: {file}")

In [ ]:
#Processing debrief data at the dyadic level 

renamed_debrief_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Renameddebriefpairs'
event_columns = [
    'Planning', 'Surprise',
    'Expectation confirmation',
    'Theory of Mind', 'Happy',
    'Frustrated'
]
result_data = []
files = [f for f in os.listdir(renamed_debrief_folder) if f.endswith('.xlsx')]
files_sorted = sorted(files, key=lambda x: int(x.split('_')[1].split('.')[0]))
for i in range(0, len(files_sorted), 2):
    file_odd = files_sorted[i]
    file_even = files_sorted[i + 1]
    dyad_num = int(file_odd.split('_')[1].split('.')[0])
    odd_df = pd.read_excel(os.path.join(renamed_debrief_folder, file_odd))
    even_df = pd.read_excel(os.path.join(renamed_debrief_folder, file_even))
    odd_df.columns = odd_df.columns.str.strip()
    even_df.columns = even_df.columns.str.strip()
    odd_df['Time (secs)'] = odd_df['Time (secs)'].str.strip()
    even_df['Time (secs)'] = even_df['Time (secs)'].str.strip()
    assert set(odd_df.columns) == set(even_df.columns), "Mismatch in columns between odd and even frames."
    for index, odd_row in odd_df.iterrows():
        for col in event_columns:
            odd_time = pd.to_datetime(odd_row['Time (secs)'], format='%H:%M:%S')
            matching_events = even_df[
             (pd.to_datetime(even_df['Time (secs)'], format='%H:%M:%S') >= (odd_time - pd.Timedelta(seconds=5))) &
             (pd.to_datetime(even_df['Time (secs)'], format='%H:%M:%S') <= (odd_time + pd.Timedelta(seconds=5))) &
             (even_df[col] == odd_row[col])
            ]
            count_matching = len(matching_events)
            result_data.append({
                'Participant': dyad_num,
                'Event Type': col,
                'Matching Count': count_matching
            })
result_df = pd.DataFrame(result_data)
result_df_pivot = result_df.pivot_table(
    index=['Participant'],
    columns='Event Type',
    values='Matching Count',
    aggfunc='sum', 
    fill_value=0
)
result_df_pivot.reset_index(inplace=True)
result_df_pivot = result_df_pivot.loc[:, (result_df_pivot != 0).any(axis=0)]
result_df_pivot = result_df_pivot.drop(columns=['Event Type'], errors='ignore')
result_df_pivot.to_csv('/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/Debriefsummaryscores/synchronycounts.csv', index=False)

In [ ]:
#Copied directly from MATLAB

import numpy as np

follow_onsets = np.array([94.8, 194.6, 419.44, 466.6, 638.44, 709.68, 885.2])
chase_onsets = np.array([5, 168.76, 556.08, 561.2, 735.12, 795.88])
in_between_onsets = np.array([0, 40, 138.8, 242.6, 464.44, 514.6, 621.2, 686.44, 767.12, 855.88])
follow_durations = np.array([44, 48, 45, 48, 48, 50, 43])
chase_durations = np.array([35, 60, 20, 60, 32, 60])
in_between_durations = np.array([5, 54.8, 29.96, 176.84, 2.16, 41.48, 17.24, 23.24, 28.76, 29.32])
follow_end_times = follow_onsets + follow_durations
chase_end_times = chase_onsets + chase_durations
in_between_end_times = in_between_onsets + in_between_durations

def check_overlaps(onsets1, end_times1, onsets2, end_times2, label1, label2):
    overlaps = []
    for i in range(len(onsets1)):
        for j in range(len(onsets2)):
            if onsets1[i] < end_times2[j] and onsets2[j] < end_times1[i]:
                overlaps.append(f"Overlap between {label1} event {i+1} and {label2} event {j+1}")
    return overlaps

follow_chase_overlaps = check_overlaps(follow_onsets, follow_end_times, chase_onsets, chase_end_times, "Follow", "Chase")
follow_in_between_overlaps = check_overlaps(follow_onsets, follow_end_times, in_between_onsets, in_between_end_times, "Follow", "In-between")
chase_in_between_overlaps = check_overlaps(chase_onsets, chase_end_times, in_between_onsets, in_between_end_times, "Chase", "In-between")
all_overlaps = follow_chase_overlaps + follow_in_between_overlaps + chase_in_between_overlaps
if all_overlaps:
    for overlap in all_overlaps:
        print(overlap)
else:
    print("No overlaps found.")

In [ ]:
#Amending participantfilesSHIFT if necessary

import pandas as pd
import os

input_folder = '/Users/macbook/Desktop/7 UCL PhD (2022-2025)/2 Minecraft fNIRS-fMRI Study/Study Files/Minecraftanalysis/Analysispipeline/fNIRSanalysispipeline/fNIRSpreprocessing/fNIRS stamps/ParticipantFilesSHIFT'

def process_file(file_path):
    df = pd.read_excel(file_path)
    df['Onset time'] = pd.to_datetime(df['Onset time'])
    df['FNIRS time'] = 5
    for i in range(1, len(df)):
        time_diff = (df.loc[i, 'Onset time'] - df.loc[i - 1, 'Onset time']).total_seconds()
        df.loc[i, 'FNIRS time'] = df.loc[i - 1, 'FNIRS time'] + time_diff
    return df

files = [f for f in os.listdir(input_folder) if f.endswith('.xlsx')]
for filename in files:
    file_path = os.path.join(input_folder, filename)
    updated_df = process_file(file_path)
    output_file_path = os.path.join(input_folder, filename)
    updated_df.to_excel(output_file_path, index=False)
    print(f"Processed and saved: {output_file_path}")